In [19]:
import pdfplumber
import csv
import re

In [20]:
column_headers = ['cip_year', 'project_type', 'source_page', 'department','project_name','project_id','start_year','end_year', 
                      'previous_appropriations', 'project_total']
years = {}
cip_year='2025-page141'

In [35]:
cleaned = []
headers = []
first_page = True

RE_DOLLARS = re.compile(r'[\$,\s]')
NON_NUMERIC = {'—', '-', 'N/A', 'NA', 'n/a', 'TBD', 'Zero', '', 'nan'}

def clean_number(s: str) -> str:
    """Normalize a currency string to plain int-string or empty."""
    s = RE_DOLLARS.sub('', s.strip())
    if s in NON_NUMERIC or s == '':
        return ''
    # Parenthesised negatives: (1234) → -1234
    if s.startswith('(') and s.endswith(')'):
        s = '-' + s[1:-1]
    try:
        return str(int(float(s)))
    except ValueError:
        return s

def extract_field(text: str, label: str) -> str:
    next_labels = r'(?:Justification|Expenditure|Operating Budget|Relationship|Schedule|Summary)'
    m = re.search(label + r'\s*:\s*(.+?)(?=' + next_labels + r'|$)', text, re.S | re.I)
    if m:
        return re.sub(r'\s+', ' ', m.group(1)).strip()
    return ''

with pdfplumber.open(r"C:\Users\vince\Documents\GitHub\CIPBD\San-Diego\PDF\\" + f"{cip_year}.pdf") as pdf:
    pages = pdf.pages
    for pg in pages:
        txt = pg.extract_text() or ''
        pg_table = pg.extract_table()
        fy_years = list(dict.fromkeys(re.findall(r'FY\s*(20\d{2})', txt)))

        if pg_table and "Duration" in txt and "Contact" in txt and "Justification" in txt:
            # Find FY year columns from header
            total_line = next((l for l in txt.splitlines() if re.match(r'^\s*Total\b', l)), None)
            if total_line:
                mid_x = pg.width / 2
                left_text  = pg.within_bbox((0,     0, mid_x,    pg.height)).extract_text() or ''
                right_text = pg.within_bbox((mid_x, 0, pg.width, pg.height)).extract_text() or ''

                # Header
                lines    = left_text.splitlines()
                department = lines[0].strip()
                name_id    = re.match(r'(.+?)\s*/\s*([A-Z]\w+)', lines[1]) if len(lines) > 1 else None
                project_name = name_id.group(1).strip() if name_id else ''
                project_id   = name_id.group(2).strip() if name_id else ''
                project_type = right_text.splitlines()[0].strip() if right_text else ''

                # Info box
                cd_m  = re.search(r'Council District\s*:\s*(.+?)(?=Community|Priority|\n)', txt)
                cp_m  = re.search(r'Community Plan(?:ning)?\s*:\s*(.+?)(?=Project|Priority|\n)', txt)
                dur_m = re.search(r'Duration\s*:\s*(\d{4})\s*[-–]\s*(\d{4})', txt)
                address_location = '; '.join(filter(None, [
                    'Council District: ' + cd_m.group(1).strip() if cd_m else '',
                    'Community Plan: '   + cp_m.group(1).strip() if cp_m else '',
                ]))
                start_year = dur_m.group(1) if dur_m else ''
                end_year   = dur_m.group(2) if dur_m else ''

                # Text fields
                description   = extract_field(left_text, 'Description')
                justification = extract_field(left_text, 'Justification')

                # Funding table
                table_section = txt[txt.find('Expenditure by Funding Source'):]
                fy_years  = list(dict.fromkeys(re.findall(r'FY\s*(20\d{2})', table_section)))
                total_line = next((l for l in txt.splitlines() if re.match(r'^\s*Total\b', l)), None)
                if total_line:
                    nums    = [clean_number(n) for n in re.findall(r'\$\s*([\d,]+|-+)', total_line)]
                    fy_nums = nums[2:-2]
                    year_cols = {f'year_{yr}': fy_nums[i] for i, yr in enumerate(fy_years) if i < len(fy_nums)}
                else:
                    nums, fy_nums, year_cols = [], [], {}

                row = {
                    'cip_year':                cip_year,
                    'project_type':            project_type,
                    'source_page':             1,
                    'department':              department,
                    'project_name':            project_name,
                    'project_id':              project_id,
                    'address_location':        address_location,
                    'start_year':              start_year,
                    'end_year':                end_year,
                    'project_description':     description,
                    'project_justification':   justification,
                    'previous_appropriations': nums[0] if nums else '',
                    'project_total':           nums[-1] if nums else '',
                }
                row.update(year_cols)
                cleaned.append(row)

for keys, values in cleaned[0].items():
    print(str(keys)+": "+str(values))
                

cip_year: 2025-page141
project_type: Bldg - Libraries
source_page: 1
department: Library
project_name: Oak Park Library
project_id: S22011
address_location: Council District: 4; Community Plan: Mid-City: Eastern Area
start_year: 2022
end_year: 2030
project_description: This project provides for the design and construction of a new library of approximately 20,000 square feet and a new book sorting facility of approximately 10,000 square feet. The library building will consist of entry/community services, technology lab, reader service area, informal reading/special feature area, reference area, multipurpose room, community room, adult/young adult area, children’s area, and staff support areas. The facility will also require a parking lot as well as building entrance and path of travel from nearby school and park areas. This project was converted from P20004 in Fiscal Year 2022.
project_justification: The existing Oak Park Library is only 5,200 square feet and is insufficient to meet the